# 10 · Data Cleaning

| Step | Action |
|------|--------|
| 1 | Load raw CSV, parse `\\N` as NaN |
| 2 | Drop rows with ATD ≤ 0 or > 120 min |
| 3 | Drop rows missing critical columns |
| 4 | Flag and impute missing distances |
| 5 | Flag missing `driver_uuid`, fill `UNKNOWN` |
| 6 | Save `cleaned.parquet` + `preprocessed.parquet` |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RAW_PATH   = Path('../data/raw/BC_A&A_with_ATD.csv')
CLEAN_PATH = Path('../data/processed/cleaned.parquet')
PREP_PATH  = Path('../data/processed/preprocessed.parquet')
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)

ATD_MAX_MIN = 120

---
## 1 · Load

In [ ]:
df = pd.read_csv(
    RAW_PATH,
    dtype={
        'region': 'category',
        'territory': 'category',
        'country_name': 'category',
        'courier_flow': 'category',
        'geo_archetype': 'category',
        'merchant_surface': 'category',
    },
    parse_dates=[
        'restaurant_offered_timestamp_utc',
        'order_final_state_timestamp_local',
        'eater_request_timestamp_local',
    ],
    na_values=['\\N', 'NULL', 'None', ''],
)

print(f'Rows    : {len(df):,}')
print(f'Columns : {df.shape[1]}')
print(f'Memory  : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
print()
print('Nulls per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])

---
## 2 · Remove Invalid ATD Rows

In [ ]:
n0 = len(df)
df = df[(df['ATD'] > 0) & (df['ATD'] <= ATD_MAX_MIN)].copy()
print(f'Removed {n0 - len(df):,} rows (ATD ≤ 0 or > {ATD_MAX_MIN} min).')
print(f'Remaining: {len(df):,}')

---
## 3 · Drop Rows Missing Critical Columns

In [ ]:
CRITICAL = [
    'ATD',
    'restaurant_offered_timestamp_utc',
    'eater_request_timestamp_local',
    'order_final_state_timestamp_local',
    'territory',
]

n0 = len(df)
df.dropna(subset=CRITICAL, inplace=True)
print(f'Removed {n0 - len(df):,} rows missing critical columns.')
print(f'Remaining: {len(df):,}')

---
## 4 · Distance: Flag + Impute

In [ ]:
df['is_distance_missing'] = df['pickup_distance'].isna().astype('int8')

for col in ['pickup_distance', 'dropoff_distance']:
    ter_median = df.groupby(
        'territory', observed=True
    )[col].transform('median')
    df[col] = df[col].fillna(ter_median).fillna(df[col].median())

print(
    f'Imputed {df["is_distance_missing"].sum():,}'
    ' rows with missing distances.'
)

---
## 5 · driver_uuid: Flag + Fill

In [ ]:
df['driver_uuid_missing'] = df['driver_uuid'].isna().astype('int8')
df['driver_uuid'] = df['driver_uuid'].fillna('UNKNOWN')
print(
    f'Unknown drivers: {df["driver_uuid_missing"].sum():,}'
    f' ({df["driver_uuid_missing"].mean()*100:.2f}%)'
)

---
## 7 · Save

In [ ]:
df.to_parquet(CLEAN_PATH, index=False, engine='pyarrow')
print(f'cleaned.parquet  → {df.shape}  ({CLEAN_PATH.stat().st_size/1024**2:.1f} MB)')

# Subset for the Streamlit dashboard (matches loader.py _BASE_COLS)
DASH_COLS = [
    'territory', 'country_name', 'workflow_uuid', 'driver_uuid',
    'delivery_trip_uuid', 'courier_flow',
    'restaurant_offered_timestamp_utc',
    'order_final_state_timestamp_local',
    'eater_request_timestamp_local',
    'geo_archetype', 'merchant_surface',
    'pickup_distance', 'dropoff_distance', 'ATD',
]
df[[c for c in DASH_COLS if c in df.columns]].to_parquet(
    PREP_PATH, index=False, engine='pyarrow'
)
print(f'preprocessed.parquet → {PREP_PATH.stat().st_size/1024**2:.1f} MB')

print(f'\nFinal shape: {df.shape}')
print(f'ATD  range: {df["ATD"].min():.1f} – {df["ATD"].max():.1f} min')
print(f'Median ATD: {df["ATD"].median():.1f} min')